# 22 - Python 3.13 New Features


## 1. Free-Threaded CPython (No-GIL) & Experimental JIT Compiler (PEP 703 & PEP 744)

Python 3.13, çok çekirdekli işlemcilerde gerçek paralel iş parçacığı (thread) yürütülmesini engelleyen Global Interpreter Lock (GIL) kilidinin isteğe bağlı olarak devre dışı bırakılabildiği serbest iş parçacıklı (`python3.13t`) derlemeyi ve kopyala-yama (copy-and-patch) tabanlı deneysel JIT derleyicisini sunar.

**Syntax**

```Python
import sys
import sysconfig

gil_disabled_build = bool(sysconfig.get_config_var('Py_GIL_DISABLED'))
is_gil_active = sys._is_gil_enabled()
```

**Mimari Özellikler ve Kontrol Mekanizmaları**

* **Free-Threaded Derleme (`--disable-gil` / `python3.13t`):** CPU yoğunluklu (CPU-bound) çoklu iş parçacığı (multithreading) iş yüklerinin tek bir süreç içinde farklı çekirdeklere paralel dağıtılmasını sağlar; çalışma zamanında `PYTHON_GIL=1` ortam değişkeni veya `-X gil=1` bayrağı ile GIL yeniden etkinleştirilebilir.
* **Çalışma Zamanı GIL Sorgusu (`sys._is_gil_enabled()`):** Mevcut yorumlayıcı örneğinde GIL kilidinin o an aktif olup olmadığını `bool` olarak döndürür.
* **Copy-and-Patch JIT Derleyici (`PYTHON_JIT=1`):** Sık çalışan (hot) bytecode yönergelerini önceden derlenmiş makine kodu şablonlarıyla birleştirerek hızlandırır; `--enable-experimental-jit` ile derlenen sürümlerde `PYTHON_JIT=1` ile etkinleştirilir.


In [ ]:
import os
import sys
import sysconfig
import time
from concurrent.futures import ThreadPoolExecutor

gil_build_flag = sysconfig.get_config_var('Py_GIL_DISABLED')
gil_enabled: bool = sys._is_gil_enabled() if hasattr(sys, '_is_gil_enabled') else True

print('Python version:', sys.version.split()[0])
print('Free-threaded build flag:', gil_build_flag)
print('Runtime GIL enabled:', gil_enabled)
print('CPU cores available:', os.cpu_count())


def cpu_bound_sum(limit: int) -> int:
    total: int = 0
    for i in range(limit):
        total += i * i
    return total


start: float = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    results: list[int] = list(pool.map(cpu_bound_sum, [2_000_000] * 4))
elapsed: float = time.perf_counter() - start

print(f'4 workers finished in {elapsed:.4f}s (sample result: {results[0]})')


## 2. Type Parameter Defaults (`[T = DefaultType]`) (PEP 696)

Python 3.12'de eklenen PEP 695 tip parametresi sözdizimi Python 3.13 ile genişletilerek `TypeVar`, `ParamSpec` ve `TypeVarTuple` parametrelerine varsayılan değer (`= DefaultType`) atanabilmesi sağlanmıştır.

**Syntax**

```Python
class Response[T = str]:
    def __init__(self, body: T) -> None:
        self.body = body


def parse_payload[T = dict[str, object]](raw: T) -> T:
    return raw
```

**Kullanım Kuralları**

* **Parametre Sıralaması:** Tıpkı fonksiyon argümanlarında olduğu gibi, varsayılan değere sahip bir tip parametresinden (`T = str`) sonra varsayılan değeri olmayan zorunlu bir tip parametresi gelemez.
* **Geriye Dönük Uyumluluk:** Mevcut bir sınıfa yeni bir tip parametresi eklendiğinde varsayılan değer tanımlamak, sınıfı eski tek parametreli biçimiyle (`Response[int]`) kullanan kodların kırılmasını önler.


In [ ]:
from typing import TypeVar


class ApiResult[T = str, E = Exception]:
    def __init__(self, data: T | None = None, error: E | None = None) -> None:
        self.data = data
        self.error = error


LegacyT = TypeVar('LegacyT', default=int)

default_res: ApiResult = ApiResult(data='OK')
custom_res: ApiResult[list[int], ValueError] = ApiResult(data=[1, 2, 3])

print('Default ApiResult:', default_res.data)
print('Custom ApiResult:', custom_res.data)
print('Legacy TypeVar default:', LegacyT.__default__)


## 3. `@warnings.deprecated` Decorator (PEP 702)

Standart kütüphanedeki `warnings` modülüne eklenen `@warnings.deprecated` dekoratörü, kullanım ömrü dolmuş (deprecated) fonksiyon, sınıf veya metotları hem çalışma zamanında `DeprecationWarning` üretecek şekilde hem de statik tip denetleyicilerin (Pyright, Mypy) editörde üzerini çizerek uyaracağı biçimde işaretler.

**Syntax**

```Python
from warnings import deprecated


@deprecated('Use connect_v2() instead')
def connect_v1() -> None:
    ...
```

**Özellikler ve Parametreler**

* **Çift Yönlü Uyarı:** Fonksiyon çağrıldığında veya sınıf örneklendiğinde otomatik olarak `DeprecationWarning` fırlatır; aynı zamanda `__deprecated__` özniteliğini ayarlayarak IDE ve statik analiz araçlarının çağrı noktalarını işaretlemesini sağlar.
* **Kategori ve Yığın Seviyesi:** Varsayılan uyarı kategorisi `DeprecationWarning` olup `category=FutureWarning` veya `stacklevel=2` gibi parametrelerle özelleştirilebilir.


In [ ]:
import warnings
from warnings import deprecated


@deprecated('legacy_loader is obsolete; use modern_loader instead.')
def legacy_loader(path: str) -> str:
    return f'Loaded {path} via v1'


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    output: str = legacy_loader('config.json')
    print('Output:', output)
    if caught:
        print('Captured warning:', caught[0].category.__name__, '->', caught[0].message)


## 4. `typing.ReadOnly` for `TypedDict` (PEP 705) & `typing.TypeIs` (PEP 742)

Python 3.13 tip sistemine `TypedDict` anahtarlarının değiştirilmesini statik olarak yasaklayan `ReadOnly[T]` belirteci ile `TypeGuard`'ın yerini alan çift yönlü tip daraltma (type narrowing) aracı `TypeIs[T]` eklenmiştir.

**Syntax**

```python
from typing import ReadOnly, TypedDict, TypeIs


class TrackMeta(TypedDict):
    id: ReadOnly[int]
    title: str


def is_str_list(val: list[object]) -> TypeIs[list[str]]:
    return all(isinstance(x, str) for x in val)
```

**`TypeIs` vs. `TypeGuard` Farkı**

* **Çift Yönlü Daraltma (`TypeIs`):** `TypeGuard[int]` yalnızca `if` bloğunun `True` dalında değişken tipini `int` yaparken `else` dalında daraltma yapmaz; `TypeIs[int]` ise `val: int | str` değişkenini `True` dalında `int`, `False` (`else`) dalında ise otomatik olarak `str` tipine daraltır.
* **Salt Okunur Sözlük Alanları (`ReadOnly`):** `TrackMeta` içindeki `id` alanı dışarıdan okunabilir ancak `meta['id'] = 2` gibi yeniden atama girişimleri tip denetleyiciler tarafından engellenir.


In [ ]:
from typing import ReadOnly, TypedDict, TypeIs


class TrackMeta(TypedDict):
    track_id: ReadOnly[int]
    title: str


def is_int(value: int | str) -> TypeIs[int]:
    return isinstance(value, int) and not isinstance(value, bool)


def format_token(token: int | str) -> str:
    if is_int(token):
        return f'ID#{token:04d}'
    return token.upper()


meta: TrackMeta = {'track_id': 101, 'title': 'Rumba Improvisation'}
meta['title'] = 'Bulerias Improvisation'

print('TrackMeta:', meta)
print('Narrowed int:', format_token(42))
print('Narrowed str:', format_token('master'))


## 5. Defined `locals()` Semantics (PEP 667), Interactive REPL & Standard Library Updates

Python 3.13 ile fonksiyon kapsamı içinde `locals()` çağrısının davranışı standartlaştırılmış, yeni renkli ve çok satırlı etkileşimli kabuk (PyREPL) devreye alınmış ve `copy.replace()` gibi kullanışlı standart kütüphane araçları eklenmiştir.

**Syntax**

```python
from copy import replace
from dataclasses import dataclass


@dataclass(frozen=True)
class Config:
    host: str
    port: int


updated = replace(Config('localhost', 8000), port=9000)
```

**Önemli Değişiklikler**

* **Tutarlı `locals()` Anlık Görüntüsü (PEP 667):** Optimize edilmiş fonksiyon çerçevelerinde `locals()` artık her çağrıda yerel değişkenlerin bağımsız bir sözlük anlık görüntüsünü (snapshot) döndürür; sözlük üzerinde yapılan değişiklikler yerel değişkenleri belirsiz biçimde bozmaz.
* **Genel `copy.replace()` Fonksiyonu:** Önceden yalnızca `dataclasses.replace` veya `namedtuple._replace` ile yapılabilen değişmez (immutable) nesne kopyalama işlemi, `__replace__` protokolünü destekleyen tüm nesneler (`dataclass`, `namedtuple`, `datetime`, `date`, `time`, `inspect.Signature` vb.) için tek bir `copy.replace()` altında birleştirilmiştir.
* **Yeni PyREPL ve Renkli İzlemeler (Tracebacks):** Çok satırlı düzenleme, geçmiş gezintisi, F1/F2/F3 kısayolları ve varsayılan renkli hata izlemeleri standart yorumlayıcıya eklenmiştir; ayrıca `dbm.sqlite3` arka ucu ve PEP 594 kapsamında 19 eski standart kütüphane modülünün (`cgi`, `telnetlib`, `sndhdr`, `imghdr` vb.) kaldırılması tamamlanmıştır.


In [ ]:
from copy import replace
from dataclasses import dataclass
from datetime import date


@dataclass(frozen=True)
class ServerConfig:
    host: str
    port: int
    ssl: bool = False


cfg = ServerConfig(host='127.0.0.1', port=8080)
prod_cfg = replace(cfg, port=443, ssl=True)

release_date = date(2024, 10, 7)
next_year_date = replace(release_date, year=2025)


def inspect_locals_snapshot() -> dict[str, object]:
    x: int = 10
    snap = locals()
    snap['x'] = 999
    return {'actual_x': x, 'snapshot_x': snap['x']}


print('Updated dataclass:', prod_cfg)
print('Updated date:', next_year_date)
print('PEP 667 locals snapshot:', inspect_locals_snapshot())
